In [15]:
import pandas as pd
import kagglehub
import shutil
import os
from pathlib import Path


In [16]:
# Download to kagglehub cache first
path = kagglehub.dataset_download("loganlauton/nba-injury-stats-1951-2023")

# Copy files to data/raw folder
dest = "../data/raw"
os.makedirs(dest, exist_ok=True)

for file in os.listdir(path):
    dest_file = os.path.join(dest, file)
    if not os.path.exists(dest_file):
        shutil.copy(os.path.join(path, file), dest)
        print(f"Copied: {file}")
    else:
        print(f"Skipped (already exists): {file}")

print("Files copied to:", dest)

Skipped (already exists): NBA Player Injury Stats(1951 - 2023).csv
Files copied to: ../data/raw


In [17]:
df_injury = pd.read_csv(f'{dest}/{file}')
df_injury.head(10)

,Unnamed: 0,Date,Team,Acquired,Relinquished,Notes
0,0,1951-12-25,Bullets,NaN,Don Barksdale,placed on IL
1,1,1952-12-26,Knicks,NaN,Max Zaslofsky,placed on IL with torn side muscle
2,2,1956-12-29,Knicks,NaN,Jim Baechtold,placed on inactive list
3,3,1959-01-16,Lakers,NaN,Elgin Baylor,player refused to play after being denied a ro...
4,4,1961-11-26,Lakers,NaN,Elgin Baylor,player reported for military duty
5,5,1962-03-24,Lakers,Elgin Baylor,NaN,player given 2-day pass from military duty
6,6,1962-03-31,Lakers,Elgin Baylor,NaN,player given weekend pass from military duty
7,7,1962-10-25,Zephyrs,NaN,Al Ferrari,placed on disabled list
8,8,1962-11-06,Zephyrs,Al Ferrari,NaN,activated from disabled list
9,9,1962-11-14,Zephyrs,NaN,Al Ferrari,placed on disabled list with knee injury


In [21]:
df_injury = pd.read_csv(f'{dest}/{file}')

# drop the junk index
df_injury = df_injury.drop(columns=["Unnamed: 0"])

# filter to your window
df_injury["Date"] = pd.to_datetime(df_injury["Date"])
df_injury = df_injury[df_injury["Date"].dt.year >= 2015]
df_injury = df_injury.reset_index(drop=True)

print(f"Shape: {df_injury.shape}")
print(f"Date range: {df_injury['Date'].min()} → {df_injury['Date'].max()}")
print(f"\nNotes value counts (top 20):\n{df_injury['Notes'].value_counts().head(20)}")
print(f"\nNull counts:\n{df_injury.isnull().sum()}")
print(f"\nSample:\n{df_injury.head(10)}")

Shape: (16316, 5)
Date range: 2015-01-01 00:00:00 → 2023-04-16 00:00:00

Notes value counts (top 20):
Notes
activated from IL                                    7724
placed on IL                                         2175
placed on IL with NBA health and safety protocols     522
placed on IL with illness                             443
placed on IL with sprained left ankle                 284
placed on IL with sprained right ankle                280
placed on IL with sore left knee                      180
placed on IL with sore right knee                     177
placed on IL for rest                                 154
placed on IL with concussion                          113
placed on IL with left knee injury                    105
placed on IL with sore lower back                      91
placed on IL with sore left ankle                      84
placed on IL with right knee injury                    79
placed on IL with sore right ankle                     71
placed on IL with left

In [20]:
df_injury.head(10)

,Date,Team,Acquired,Relinquished,Notes
0,2015-01-01,Kings,NaN,Omri Casspi,placed on IL with left knee injury
1,2015-01-02,Bucks,NaN,Larry Sanders (b. 1988-11-21),placed on IL
2,2015-01-02,Knicks,NaN,Tim Hardaway Jr.,placed on IL with concussion
3,2015-01-02,Magic,NaN,Maurice Harkless / Moe Harkless,placed on IL
4,2015-01-02,Nets,NaN,Cory Jefferson,placed on IL
5,2015-01-02,Pistons,NaN,Spencer Dinwiddie,placed on IL
6,2015-01-02,Suns,NaN,Tyler Ennis,placed on IL
7,2015-01-03,Blazers,NaN,Victor Claver,placed on IL
8,2015-01-03,Bulls,NaN,Mike Dunleavy Jr.,placed on IL with right ankle injury
9,2015-01-03,Jazz,NaN,Patrick Christopher,placed on IL with right knee injury


In [18]:
df_injury.dtypes

Unnamed: 0      int64
Date              str
Team              str
Acquired          str
Relinquished      str
Notes             str
dtype: object

### Setup - Creating Data Directories

In [ ]:
Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Download to kagglehub cache first
path = kagglehub.dataset_download("loganlauton/nba-injury-stats-1951-2023")

# Copy files to data/raw folder
dest = "../data/raw"
os.makedirs(dest, exist_ok=True)

### Imports & Directory Setup


In [ ]:
from nba_api.stats.endpoints import PlayerGameLogs
from pathlib import Path
import pandas as pd
import time

# Directory Setup
RAW_DIR = Path("../data/raw/game_logs")
RAW_DIR.mkdir(parents=True, exist_ok=True)

seasons = [
    "2015-16", "2016-17", "2017-18", "2018-19",
    "2019-20", "2020-21", "2021-22", "2022-23"
]

season_types = ["Regular Season", "Playoffs"]

### Step 4 - Fetching player logs from the 2015-2016 season through the 2022-23 season. A total of eight seasons.

In [ ]:
def fetch_season_logs(season, season_type, retries=3):
    for attempt in range(retries):
        try:
            logs = PlayerGameLogs(
                season_nullable=season,
                season_type_nullable=season_type,
                timeout=120,
            )
            return logs.get_data_frames()[0]
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                wait = (attempt + 1) * 5
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
    return None


for season in seasons:
    for season_type in season_types:
        label = season_type.lower().replace(" ", "_")
        out_path = RAW_DIR / f"player_game_logs_{season.replace('-', '_')}_{label}.csv"

        if out_path.exists():
            print(f"Skipping {season} {season_type} — already exists")
            continue

        print(f"Fetching {season} {season_type}...")
        df = fetch_season_logs(season, season_type)

        if df is None:
            print(f"  ❌ All retries failed for {season} {season_type}")
        elif df.empty:
            print(f"  ⚠️ No data for {season} {season_type}")
        else:
            df.to_csv(out_path, index=False)
            print(f"  ✅ {len(df)} rows → {out_path.name}")

        time.sleep(3)

print("Done!")

Fetching 2015-16 Regular Season...
  ✅ 26078 rows → player_game_logs_2015_16_regular_season.csv
Fetching 2015-16 Playoffs...
  ✅ 1900 rows → player_game_logs_2015_16_playoffs.csv
Fetching 2016-17 Regular Season...
  ✅ 26139 rows → player_game_logs_2016_17_regular_season.csv
Fetching 2016-17 Playoffs...
  ✅ 1737 rows → player_game_logs_2016_17_playoffs.csv
Fetching 2017-18 Regular Season...
  ✅ 26107 rows → player_game_logs_2017_18_regular_season.csv
Fetching 2017-18 Playoffs...
  ✅ 1729 rows → player_game_logs_2017_18_playoffs.csv
Fetching 2018-19 Regular Season...
  ✅ 26101 rows → player_game_logs_2018_19_regular_season.csv
Fetching 2018-19 Playoffs...
  ✅ 1761 rows → player_game_logs_2018_19_playoffs.csv
Fetching 2019-20 Regular Season...
  ✅ 22393 rows → player_game_logs_2019_20_regular_season.csv
Fetching 2019-20 Playoffs...
  ✅ 1694 rows → player_game_logs_2019_20_playoffs.csv
Fetching 2020-21 Regular Season...
  ✅ 23054 rows → player_game_logs_2020_21_regular_season.csv
Fetching 

### Step 5 - Combine and save raw game logs

In [28]:
import glob

files = glob.glob("../data/raw/game_logs/**/*.csv", recursive=True)

dfs = []
for f in files:
    temp = pd.read_csv(f)
    temp["is_playoff"] = 1 if "playoff" in f else 0
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df.to_csv("../data/processed/player_game_logs_all.csv", index=False)
print(df.shape)

(216109, 71)


In [33]:
df_game_logs = pd.read_csv("../data/processed/player_game_logs_all.csv")
print(df_game_logs.shape)        # row and column count
print(df_game_logs.head())       # first 5 rows
print(df_game_logs.dtypes)       # column types

(216109, 71)
  SEASON_YEAR  PLAYER_ID         PLAYER_NAME      NICKNAME     TEAM_ID  \
0     2016-17     203999        Nikola Jokić        Nikola  1610612743   
1     2016-17     201935        James Harden         James  1610612745   
2     2016-17    1626157  Karl-Anthony Towns  Karl-Anthony  1610612750   
3     2016-17     203932        Aaron Gordon         Aaron  1610612753   
4     2016-17     202331         Paul George          Paul  1610612754   

  TEAM_ABBREVIATION               TEAM_NAME   GAME_ID            GAME_DATE  \
0               DEN          Denver Nuggets  21601225  2017-04-12T00:00:00   
1               HOU         Houston Rockets  21601224  2017-04-12T00:00:00   
2               MIN  Minnesota Timberwolves  21601224  2017-04-12T00:00:00   
3               ORL           Orlando Magic  21601217  2017-04-12T00:00:00   
4               IND          Indiana Pacers  21601226  2017-04-12T00:00:00   

       MATCHUP  ... PTS_RANK  PLUS_MINUS_RANK  NBA_FANTASY_PTS_RANK  DD2_

In [30]:
print(df_game_logs.isnull().sum()) # check for missing values

SEASON_YEAR               0
PLAYER_ID                 0
PLAYER_NAME               0
NICKNAME                  0
TEAM_ID                   0
                         ..
WNBA_FANTASY_PTS_RANK     0
AVAILABLE_FLAG           90
MIN_SEC                   0
TEAM_COUNT                0
is_playoff                0
Length: 71, dtype: int64


In [39]:
print(df_game_logs[df_game_logs['is_playoff'] == 1].shape)

print(df_game_logs[df_game_logs['is_playoff'] == 0].shape)

(14304, 71)
(201805, 71)
